# RO-050 Potential fields: the numbers

Every number quoted in the Note, computed here. The model lives in `images/pf.py` (the room of RO-009 in the disk robot's C-space, from `images/cspace.py`).

In [1]:
import sys
sys.path.insert(0, 'images')
import numpy as np
from scipy.optimize import root
import pf, cspace as c
np.set_printoptions(precision=4, suppress=True)
print('zeta', pf.ZETA, 'eta', pf.ETA, 'Q*', pf.QSTAR, 'step factor', pf.STEP, 'longest step', pf.MAXSTEP)
print('goal', pf.GOAL, 'starts', pf.START, pf.START2, 'obstacles', len(pf.OBS))

zeta 1.0 eta 0.02 Q* 0.5 step factor 0.01 longest step 0.05
goal [3.6 4. ] starts [2. 1.] [4.  0.5] obstacles 9


## 2. The attractive potential

In [2]:
q = pf.START
print('U_att at (2, 1):', pf.u_att(q))
print('gradient of U_att at (2, 1):', pf.ZETA * (q - pf.GOAL))
s = -pf.STEP * pf.grad(q)
print('first step:', s, 'length', np.hypot(*s), '-> new point', q + s)
eta, pf.ETA = pf.ETA, 0.0
P = pf.descend(pf.START); pf.ETA = eta
print('attraction only: path', pf.path_length(P), '+ last gap', np.hypot(*(P[-1] - pf.GOAL)), '(the straight 3.4 m)')


U_att at (2, 1): 5.78
gradient of U_att at (2, 1): [-1.6 -3. ]
first step: [0.016 0.03 ] length 0.033999999999999996 -> new point [2.016 1.03 ]
attraction only: path 3.380198478986543 + last gap 0.01980152101345708 (the straight 3.4 m)


## 3. The repulsive potential and the total

In [3]:
for d in (0.6, 0.5, 0.25, 0.157, 0.1):
    push = pf.ETA * (1 / d - 1 / pf.QSTAR) / d**2 if d <= pf.QSTAR else 0.0
    print(f'd = {d}: U_rep = {pf.u_rep_one(d):.4f}, push = {push:.4f}')
for name, s in (('(2, 1)', pf.START), ('(4.0, 0.5)', pf.START2)):
    print(name, 'U =', round(pf.U(s), 4), 'grad =', pf.grad(s), 'clearances within Q*:',
          [round(d, 3) for d, _ in pf.clearance_vec(s) if d <= pf.QSTAR])
PA, PB = pf.descend(pf.START), pf.descend(pf.START2)
print('from (2, 1):', len(PA) - 1, 'steps, path', round(pf.path_length(PA), 3), 'm, ends', PA[-1],
      'closest clearance', round(pf.min_clearance(PA), 3))
print('from (4.0, 0.5):', len(PB) - 1, 'steps, ends', PB[-1], 'U', round(pf.U(PB[-1]), 4),
      'distance to door', round(float(np.hypot(*(PB[-1] - pf.GOAL))), 3))

d = 0.6: U_rep = 0.0000, push = 0.0000
d = 0.5: U_rep = 0.0000, push = 0.0000
d = 0.25: U_rep = 0.0400, push = 0.6400
d = 0.157: U_rep = 0.1909, push = 3.5453
d = 0.1: U_rep = 0.6400, push = 16.0000
(2, 1) U = 5.78 grad = [-1.6 -3. ] clearances within Q*: []
(4.0, 0.5) U = 6.285 grad = [ 0.4 -3.5] clearances within Q*: [np.float64(0.25), np.float64(0.25)]


from (2, 1): 705 steps, path 3.473 m, ends [3.6    3.9801] closest clearance 0.278
from (4.0, 0.5): 598 steps, ends [3.601  0.5932] U 6.0031 distance to door 3.407


## 4. The dent: forces, critical points, basin

In [4]:
trap = root(pf.grad, (3.6, 0.59), tol=1e-14).x
print('dent', trap, 'U', pf.U(trap))
print('pull of the door', pf.ZETA * (pf.GOAL - trap))
for d, n in pf.clearance_vec(trap):
    if d <= pf.QSTAR:
        print('  clearance', round(d, 4), 'direction', n.round(3), 'push', round(pf.ETA * (1/d - 1/pf.QSTAR) / d**2, 4),
              'U_rep', round(pf.u_rep_one(d), 4))
def hess(x, h=1e-5):
    H = np.array([(pf.grad(x + h*e) - pf.grad(x - h*e)) / (2*h) for e in np.eye(2)])
    return np.linalg.eigvalsh((H + H.T) / 2)
print('dent: curvatures (eigenvalues of the Hessian)', hess(trap))
for s0 in pf.SADDLES:
    x = root(pf.grad, s0, tol=1e-14).x
    print('saddle', x.round(3), 'U', round(pf.U(x), 3), 'curvatures', hess(x).round(2))

dent [3.6    0.5932] U 6.0031220873940585
pull of the door [0.     3.4068]
  clearance 0.1568 direction [ 0. -1.] push 3.5619 U_rep 0.1917
  clearance 0.3432 direction [0. 1.] push 0.1551 U_rep 0.0083
dent: curvatures (eigenvalues of the Hessian) [ 1.     81.8776]
saddle [2.924 0.601] U 6.206 curvatures [-7.79 82.63]
saddle [5.167 0.632] U 7.105 curvatures [-7.85 86.99]
saddle [0.521 2.842] U 5.64 curvatures [-7.56 90.56]


In [5]:
step = 0.1
S = [np.array([x, y]) for y in np.arange(step/2, c.H, step) for x in np.arange(step/2, c.W, step)]
S = [q for q in S if pf.free(q)]
E = [pf.descend(q, n_max=3000, tol=1e-4)[-1] for q in S]
stuck = np.array([np.hypot(*(e - pf.TRAP)) < 0.1 for e in E])
door = np.array([np.hypot(*(e - pf.GOAL)) < 0.1 for e in E])
S = np.array(S)
print('starts', len(S), 'in the dent', stuck.sum(), round(100 * stuck.mean(), 1), 'percent; at the door', door.sum())
print('dent basin x from', S[stuck, 0].min(), 'to', S[stuck, 0].max())

starts 1529 in the dent 118 7.7 percent; at the door 1411
dent basin x from 2.95 to 5.250000000000001


## 5. Randomized potential field

In [6]:
parts = pf.randomized(pf.START2, np.random.default_rng(7))
for m, p in parts:
    print(m, len(p) - 1, 'steps, ends', p[-1].round(3), 'U', round(pf.U(p[-1]), 4), 'length', round(pf.path_length(p), 2))
print('total', round(pf.path_length(np.vstack([p for _, p in parts])), 2))
L_short, P_short = pf.shortest_route()
print('shortest route from (4.0, 0.5):', round(L_short, 3))

descend 598 steps, ends [3.601 0.593] U 6.0031 length 0.48
walk 384 steps, ends [4.901 0.493] U 7.0755 length 27.15
descend 715 steps, ends [3.601 0.593] U 6.0031 length 1.36
walk 132 steps, ends [2.701 0.693] U 6.0022 length 9.33
descend 715 steps, ends [3.6  3.98] U 0.0148 length 3.65
total 41.97


shortest route from (4.0, 0.5): 4.547


In [7]:
W, L, OK, BT, first, left = [], [], [], 0, [], []
for seed in range(200):
    parts = pf.randomized(pf.START2, np.random.default_rng(seed))
    allp = np.vstack([p for _, p in parts])
    OK.append(np.hypot(*(allp[-1] - pf.GOAL)) <= 0.05)
    w = [p for m, p in parts if m == 'walk']
    W.append(len(w)); BT += sum(m == 'backtrack' for m, _ in parts); L.append(pf.path_length(allp))
    left.append(w[-1][-1][0] < 3.6)
W, L = np.array(W), np.array(L)
print('reached', np.mean(OK), '| walks: mean', W.mean(), 'max', W.max(), 'first-walk share', np.mean(W == 1),
      '| backtracks', BT, '| length median', round(np.median(L), 1), '10th', round(np.percentile(L, 10), 1),
      '90th', round(np.percentile(L, 90), 1), '| escaped left', np.mean(left))

reached 1.0 | walks: mean 3.785 max 22 first-walk share 0.315 | backtracks 1 | length median 46.1 10th 16.3 90th 168.0 | escaped left 0.895


## 6. Harmonic potential

In [8]:
xs, ys, fr, g, phi = pf.harmonic_solve()
print('nodes', fr.size, 'free', fr.sum(), 'goal node', g)
val = lambda p: phi[int(round(p[1] / pf.H_GRID)), int(round(p[0] / pf.H_GRID))]
for name, s in (('(2, 1)', pf.START), ('(4.0, 0.5)', pf.START2)):
    P, ok = pf.grid_descent(phi, fr, s)
    print(name, '1 - phi =', 1 - val(s), '| reached', ok, len(P) - 1, 'steps', round(pf.path_length(P), 3), 'm')
gap = 1 - phi[fr]; print('smallest 1 - phi over free nodes', gap[gap > 0].min())
ny, nx = fr.shape
none_lower = 0
for jj, ii in zip(*np.nonzero(fr)):
    if (jj, ii) == g:
        continue
    nb4 = [phi[a, b] if (0 <= a < ny and 0 <= b < nx and fr[a, b]) else 1.0
           for a, b in ((jj - 1, ii), (jj + 1, ii), (jj, ii - 1), (jj, ii + 1))]
    none_lower += min(nb4) >= phi[jj, ii]
print('free points other than the door:', fr.sum() - 1, '| with no strictly lower neighbour:', none_lower)
j, i = int(round(3.6 / 0.05)), int(round(3.5 / 0.05))
nb = [phi[j, i-1], phi[j, i+1], phi[j-1, i], phi[j+1, i]]
print('node (3.5, 3.6):', round(phi[j, i], 5), 'neighbours left, right, below, above', [round(v, 4) for v in nb],
      'average', round(np.mean(nb), 5))
_, _, _, _, out = pf.gauss_seidel(sweeps=(1, 10, 100, 1000, 5000))
for k, Pk in out.items():
    print('after', k, 'sweeps: largest error', f'{np.abs(Pk - phi)[fr].max():.1e}')
print('node updates for 5,000 sweeps:', 5000 * (fr.sum() - 1))

nodes 11421 free 5840 goal node (80, 72)
(2, 1) 1 - phi = 4.832845466207214e-05 | reached True 60 steps 3.746 m
(4.0, 0.5) 1 - phi = 1.9324719602309415e-09 | reached True 91 steps 5.606 m
smallest 1 - phi over free nodes 2.298161660974074e-14
free points other than the door: 5839 | with no strictly lower neighbour: 0
node (3.5, 3.6): 0.93736 neighbours left, right, below, above [np.float64(0.9429), np.float64(0.9336), np.float64(0.9482), np.float64(0.9247)] average 0.93736


after 1 sweeps: largest error 3.3e-01
after 10 sweeps: largest error 9.9e-02
after 100 sweeps: largest error 6.2e-03
after 1000 sweeps: largest error 1.9e-05
after 5000 sweeps: largest error 4.2e-11
node updates for 5,000 sweeps: 29195000


In [9]:
v = np.ones(11); v[-1] = 0
v[9] = 0.5 * (v[8] + v[10]); print('corridor, first update of node 9:', v[9])
v = np.ones(11); v[-1] = 0
for k in range(1, 81):
    for i in range(1, 10):
        v[i] = 0.5 * (v[i-1] + v[i+1])
    if k in (5, 80): print('after sweep', k, v.round(3))

corridor, first update of node 9: 0.5
after sweep 5 [1.    1.    1.    1.    1.    0.969 0.875 0.711 0.492 0.246 0.   ]
after sweep 80 [1.  0.9 0.8 0.7 0.6 0.5 0.4 0.3 0.2 0.1 0. ]


## 7. Navigation function (Rimon-Koditschek) on the sphere world

In [10]:
print('sphere world: disk', pf.SW_C0, pf.SW_R0, 'obstacles', [(cc, r) for cc, r in pf.SW_OBS])
for k in (1, 1.25, 1.5, 1.75, 1.9, 2, 3, 4, 6):
    P = pf.rk_descend(pf.START2, k)
    print(f'kappa {k}: phi(start) = {float(pf.rk(pf.START2, k)):.6f}, ends {P[-1].round(2)}, '
          f'reaches door {np.hypot(*(P[-1] - pf.GOAL)) < 0.03}, steps {len(P) - 1}')
lo, hi = 1.95, 2.0
for _ in range(14):
    m = (lo + hi) / 2
    P = pf.rk_descend(pf.START2, m)
    lo, hi = (lo, m) if np.hypot(*(P[-1] - pf.GOAL)) < 0.03 else (m, hi)
print('smallest kappa that reaches the door from (4.0, 0.5): between', round(lo, 4), 'and', round(hi, 4))
for k in (1, 2):
    found = []
    rng = np.random.default_rng(0)
    for x0 in rng.uniform([0.2, -1.3], [6.8, 5.3], (400, 2)):
        if not np.isfinite(pf.rk(x0, k)):
            continue
        s = root(lambda q: pf.rk_grad(q, k), x0, method='lm', tol=1e-15)
        x = s.x
        if not np.isfinite(pf.rk(x, k)) or np.hypot(*pf.rk_grad(x, k)) > 1e-9 or any(np.hypot(*(x - f)) < 1e-3 for f, _ in found):
            continue
        H = np.array([(pf.rk_grad(x + 1e-6*e, k) - pf.rk_grad(x - 1e-6*e, k)) / 2e-6 for e in np.eye(2)])
        ev = np.linalg.eigvalsh((H + H.T) / 2)
        found.append((x, 'minimum' if ev.min() > 0 else 'saddle'))
    print('kappa', k, [(f[0].round(3), f[1]) for f in found])

sphere world: disk [3.5 2. ] 3.3 obstacles [(array([4.2, 1.9]), 1.3), (array([1., 3.]), 0.35), (array([6. , 0.8]), 0.35)]
kappa 1: phi(start) = 0.073645, ends [1.76 0.19], reaches door False, steps 294


kappa 1.25: phi(start) = 0.195310, ends [1.72 0.33], reaches door False, steps 298
kappa 1.5: phi(start) = 0.363089, ends [1.7 0.5], reaches door False, steps 302


kappa 1.75: phi(start) = 0.543958, ends [1.7  0.73], reaches door False, steps 310
kappa 1.9: phi(start) = 0.644511, ends [1.73 0.92], reaches door False, steps 319


kappa 2: phi(start) = 0.704716, ends [3.59 4.  ], reaches door True, steps 682


kappa 3: phi(start) = 0.974169, ends [3.58 3.99], reaches door True, steps 603
kappa 4: phi(start) = 0.998361, ends [3.59 3.99], reaches door True, steps 550


kappa 6: phi(start) = 0.999993, ends [3.59 3.99], reaches door True, steps 495


smallest kappa that reaches the door from (4.0, 0.5): between 1.9832 and 1.9832


kappa 1 [(array([1.764, 0.186]), 'minimum'), (array([3.6, 4. ]), 'minimum'), (array([2.108, 2.111]), 'saddle'), (array([5.463, 1.105]), 'saddle'), (array([6.401, 0.602]), 'saddle')]


/home/anshu/campusx/RO/navigation/06-grid-sampling-and-car-like-planning/RO-050-potential-fields/images/pf.py:272: RuntimeWarning: invalid value encountered in power
  return dgam * S ** (-1 / kappa) - (1 / kappa) * gam * S ** (-1 / kappa - 1) * (kappa * gam ** (kappa - 1) * dgam + dbeta)


kappa 2 [(array([6.396, 0.595]), 'saddle'), (array([3.6, 4. ]), 'minimum'), (array([0.528, 3.133]), 'saddle'), (array([5.44, 1.09]), 'saddle')]
